# 01 · Dataset Quality & Exploratory Data Analysis
Fashion-MNIST (Xiao et al., 2017): 70,000 28×28 grayscale clothing images, 10 classes.

Every plot is produced by `src/eda.py`; this notebook narrates the findings.

In [ ]:
import sys, warnings; sys.path.insert(0, '..'); warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
from src.config import *
from src.data_loader import load_raw, to_unit, make_splits
from src.plotting import set_style
set_style()
def show(name, w=900): display(Image(filename=str(FIG_DIR / f'{name}.png'), width=w))

## 1. Load & audit
IDX files are read directly (`src/data_loader.py`). We check shape, value range, missing values, blank images, exact duplicates and train/test leakage.

In [ ]:
Xtr_u8, ytr, Xte_u8, yte = load_raw()
from src import eda
eda.quality_report(Xtr_u8, ytr, Xte_u8, yte)

**Finding:** the data is clean - no missing/blank/duplicate images and zero train-test overlap. The only preprocessing step is scaling to [0,1].

## 2. Class balance

In [ ]:
show('eda_01_class_distribution', 700); pd.read_csv(TABLE_DIR/'eda_class_distribution.csv')

Perfectly balanced (6,000 / 1,000 per class) -> accuracy is a valid headline metric, but we still report macro-F1 and per-class recall.

## 3. What do the images look like?

In [ ]:
show('eda_02_sample_grid', 650); show('eda_12_ambiguous_group', 650)

The four upper-body classes (T-shirt/top, Pullover, Coat, Shirt) are visually near-identical silhouettes -> expected hard group.

## 4. Class means, intensity and shape statistics

In [ ]:
show('eda_03_mean_std_images'); show('eda_04_intensity_shape_boxplots'); pd.read_csv(TABLE_DIR/'eda_class_summary_stats.csv', index_col=0)

In [ ]:
show('eda_05_pixel_histogram')

About 50 % of pixels are exactly zero (background): the data is sparse. Footwear has small height / large width; trousers are narrow; bags have the widest variance in size.

## 5. Which pixels matter?

In [ ]:
show('eda_06_pixel_variance_fscore', 1000)

The border pixels have ~zero variance and ~zero F-score: they carry no information. Discriminative pixels lie at the centre and along the silhouette edges.

## 6. Is there spatial structure that flattening destroys?

In [ ]:
show('eda_07_spatial_correlation', 600); pd.read_csv(TABLE_DIR/'eda_class_pair_distance.csv').head(8)

Adjacent pixels correlate at r≈0.75, far above the shuffled null (≈0.16). Flattening treats pixel 5 and pixel 6 like any two unrelated pixels, which is the motivation for neighbourhood-based descriptors (HOG).

In [ ]:
show('eda_08_class_mean_distance', 650)

Mean-image distances predict the confusion structure: **Pullover-Coat-Shirt-T-shirt** are the closest pairs.

## 7. Intrinsic dimensionality & embeddings

In [ ]:
show('eda_09_pca'); show('eda_10_tsne', 650); pd.read_csv(TABLE_DIR/'eda_pca_components.csv')

24 principal components explain 80 % and 187 explain 95 % of the variance (of 784) -> strong redundancy. t-SNE shows footwear, trousers and bags as clean clusters while the upper-body classes overlap.

## 8. Train/test shift & orientation

In [ ]:
show('eda_11_train_test_shift', 800); pd.read_csv(TABLE_DIR/'eda_orientation_check.csv', index_col=0)

KS test: no shift between train and test. Images are upright (footwear/bags heavier at the bottom, tops/trousers heavier at the top), so HOG can be computed directly without rotating.

### EDA conclusions
1. Clean, balanced data; no cleaning required.
2. Strong spatial structure + sparsity -> neighbourhood descriptors are justified.
3. Upper-body group is the predicted difficulty.
4. Heavy pixel redundancy -> PCA/HOG compress without losing the class signal.